# Exercício 5 — sem ecrã (LCEL + *prompt templates* + *fallback* de modelos)

`ChatPromptTemplate`, operador **`|`** (*RunnableSequence*), `StrOutputParser`, `partial`, `RunnablePassthrough.assign` e `RunnableParallel`.

**Fallback em tempo real:** `make_gemini_chat_with_runtime_fallback` usa `with_fallbacks` do LangChain — **em cada** `invoke` da cadeia, se o modelo actual der **429** / `RESOURCE_EXHAUSTED` (ou outro erro recuperável), tenta-se o **seguinte** da lista automaticamente. Variáveis: **`GEMINI_MODEL_EX05`** ou **`GEMINI_MODEL`**, **`GEMINI_MODEL_FALLBACKS`** (CSV). Opcional: **`LLM_FALLBACK_SKIP_SMOKE_TEST`** só afecta `pick_working_*` (diagnóstico), não esta cadeia.

**Listar modelos:** células abaixo (Gemini; DeepSeek se existir `DEEPSEEK_API_KEY`) para copiar nomes exactos para o `.env`.

**Docker:** `./run.sh`. Chave **`GOOGLE_API_KEY`** na raiz. Ver **GUIA §9.2**.


In [1]:
import os
import sys
from pathlib import Path

key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
if not key:
    raise RuntimeError("Defina GOOGLE_API_KEY (`.env` na raiz ou Docker).")
os.environ.setdefault("GOOGLE_API_KEY", key)

_exercicios = Path.cwd().resolve().parent
if str(_exercicios) not in sys.path:
    sys.path.insert(0, str(_exercicios))

from lib_llm_fallback import (
    gemini_model_candidates,
    list_deepseek_model_ids,
    list_gemini_text_models,
    make_gemini_chat_with_runtime_fallback,
)

candidatos_gemini = gemini_model_candidates(ex05=True)
print("Ordem de tentativas Gemini:", ", ".join(candidatos_gemini), flush=True)


Ordem de tentativas Gemini: gemini-2.0-flash, gemini-2.5-flash, gemini-2.0-flash-lite, gemini-2.5-flash-lite


## Listar modelos (evita `Error calling model '…'`)

- **Gemini:** cliente Google GenAI (`client.models.list()`), filtrado para geração de texto.
- **DeepSeek:** `GET /v1/models` na `DEEPSEEK_API_BASE` — alinhado com o exercício 4.


In [2]:
try:
    nomes = list_gemini_text_models()
    print(f"Gemini — {len(nomes)} modelo(s). Primeiros 30:")
    for n in nomes[:30]:
        print(" ", n)
    if len(nomes) > 30:
        print("  …")
except Exception as e:
    print(f"Não foi possível listar Gemini: {e}", flush=True)

dk = (os.environ.get("DEEPSEEK_API_KEY") or "").strip()
if dk:
    try:
        base = (os.environ.get("DEEPSEEK_API_BASE") or "https://api.deepseek.com").strip().rstrip("/")
        ids = list_deepseek_model_ids(api_key=dk, base_url=base)
        print(f"\nDeepSeek — {len(ids)} id(s). Exemplos:")
        for n in ids[:20]:
            print(" ", n)
    except Exception as e:
        print(f"DeepSeek list: {e}", flush=True)
else:
    print("\n(Defina DEEPSEEK_API_KEY para listar modelos DeepSeek.)", flush=True)


Gemini — 25 modelo(s). Primeiros 30:
  gemini-2.0-flash
  gemini-2.0-flash-001
  gemini-2.0-flash-lite
  gemini-2.0-flash-lite-001
  gemini-2.5-computer-use-preview-10-2025
  gemini-2.5-flash
  gemini-2.5-flash-image
  gemini-2.5-flash-lite
  gemini-2.5-flash-native-audio-latest
  gemini-2.5-flash-native-audio-preview-09-2025
  gemini-2.5-flash-native-audio-preview-12-2025
  gemini-2.5-pro
  gemini-3-flash-preview
  gemini-3-pro-image-preview
  gemini-3-pro-preview
  gemini-3.1-flash-image-preview
  gemini-3.1-flash-lite-preview
  gemini-3.1-flash-live-preview
  gemini-3.1-pro-preview
  gemini-3.1-pro-preview-customtools
  gemini-flash-latest
  gemini-flash-lite-latest
  gemini-pro-latest
  gemini-robotics-er-1.5-preview
  gemini-robotics-er-1.6-preview

DeepSeek — 2 id(s). Exemplos:
  deepseek-v4-flash
  deepseek-v4-pro


## LLM Gemini com *fallback* em **cada** pedido (`with_fallbacks`)

`make_gemini_chat_with_runtime_fallback` empilha vários `ChatGoogleGenerativeAI`; o LangChain tenta o seguinte **no mesmo** `chain.invoke(...)` quando o anterior falha (ex.: **429** no `gemini-2.0-flash` e sucesso no `gemini-2.5-flash`). Se **todos** falharem, espere ou veja [limites](https://ai.google.dev/gemini-api/docs/rate-limits).


In [3]:
llm = make_gemini_chat_with_runtime_fallback(candidatos_gemini, temperature=0.3)
print(
    "LLM com fallback em tempo de execução (ordem por invoke):",
    ", ".join(candidatos_gemini),
    flush=True,
)


LLM com fallback em tempo de execução (ordem por invoke): gemini-2.0-flash, gemini-2.5-flash, gemini-2.0-flash-lite, gemini-2.5-flash-lite


## 1. Cadeia LCEL: `prompt | modelo | parser`


In [4]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "És um formador breve em português europeu. Usa no máximo 3 frases."),
        ("human", "Explica o conceito: {topico}"),
    ]
)

chain = prompt | llm | StrOutputParser()
print(chain.invoke({"topico": "o operador pipe em LangChain LCEL"}))


Em LangChain LCEL, o operador pipe (`|`) permite encadear componentes (prompts, modelos, etc.) para criar fluxos de trabalho. Ele direciona a saída de um componente como entrada para o seguinte, simplificando a construção de pipelines complexos. Pensa nele como um "e então faz isto" para os teus componentes.


## 2. `partial` — variáveis fixas no template


In [5]:
prompt_tom = ChatPromptTemplate.from_messages(
    [
        ("system", "Responde sempre em {lingua}, tom: {tom}."),
        ("human", "{pedido}"),
    ]
)
prompt_curto = prompt_tom.partial(lingua="português europeu", tom="didáctico e calmo")

chain_tom = prompt_curto | llm | StrOutputParser()
print(chain_tom.invoke({"pedido": "O que é um ChatPromptTemplate?"}))


Com certeza! Vamos explorar o conceito de `ChatPromptTemplate` de uma forma clara e didática.

### O que é um ChatPromptTemplate?

Em essência, um **`ChatPromptTemplate`** é um modelo estruturado para criar *prompts* (instruções ou perguntas) que são especificamente desenhados para **modelos de linguagem grandes (LLMs) conversacionais** ou "modelos de chat".

A principal razão para a sua existência é que os modelos de chat modernos não funcionam apenas com uma única string de texto como entrada (como alguns modelos mais antigos). Em vez disso, eles esperam uma **sequência de mensagens**, onde cada mensagem tem um **papel (role)** distinto.

Pense nele como um **guião** ou um **formulário pré-definido** para uma conversa, onde podes preencher os espaços em branco com informações dinâmicas.

#### Porquê usar um ChatPromptTemplate?

1.  **Modelos de Chat vs. Modelos de Texto:**
    *   **Modelos de Texto (antigos):** Recebiam um bloco de texto e continuavam a gerar texto. Ex: "Traduz esta

## 3. `RunnablePassthrough.assign` — enriquecer o dicionário de entrada


In [6]:
from langchain_core.runnables import RunnablePassthrough

prompt_ctx = ChatPromptTemplate.from_messages(
    [
        ("system", "És um mentor de estudo. Sê concreto."),
        (
            "human",
            "Contexto do aluno: {contexto_aluno}\n\nTarefa: {tarefa}",
        ),
    ]
)

def montar_contexto(topico: str) -> str:
    return f"Aluno a estudar «{topico}» pela primeira vez."

chain_ctx = (
    RunnablePassthrough.assign(
        contexto_aluno=lambda x: montar_contexto(x["topico"]),
        tarefa=lambda x: f"Dá um exemplo mínimo relacionado com «{x['topico']}».",
    )
    | prompt_ctx
    | llm
    | StrOutputParser()
)

print(chain_ctx.invoke({"topico": "RunnableSequence"}))


Olá! Que bom que estás a mergulhar em `RunnableSequence`. É uma das peças mais fundamentais e poderosas do LangChain para construir aplicações.

Vamos a um exemplo mínimo e concreto.

---

### O que é `RunnableSequence`?

Imagina que tens uma linha de montagem. Cada estação faz uma tarefa específica e passa o resultado para a próxima estação. `RunnableSequence` é exatamente isso: uma forma de **encadear diferentes componentes** (chamados "runnables") onde a saída de um se torna a entrada do próximo.

É a forma mais comum de construir "chains" (cadeias) no LangChain.

---

### Exemplo Mínimo: Fato Curioso sobre um Tópico

Vamos criar uma sequência que:
1.  Recebe um **tópico** (ex: "gatos").
2.  Cria um **prompt** para um modelo de linguagem (LLM) com esse tópico.
3.  Envia o prompt para o **LLM** para obter uma resposta.
4.  Extrai o **texto** da resposta do LLM.

**Pré-requisitos:**
Certifica-te de que tens as bibliotecas instaladas e a tua chave da OpenAI configurada:

```bash
pip in

## 4. `RunnableParallel` — vários ramos com o mesmo input


In [7]:
from langchain_core.runnables import RunnableParallel

p_directo = ChatPromptTemplate.from_messages(
    [("human", "Define numa frase: {termo}")]
)
p_aluno = ChatPromptTemplate.from_messages(
    [
        (
            "human",
            "Explica como a um aluno do 9.º ano, numa frase: {termo}",
        )
    ]
)

ramos = RunnableParallel(
    definicao=p_directo | llm | StrOutputParser(),
    explicacao_simples=p_aluno | llm | StrOutputParser(),
)

termo = "cadeia de prompts (LCEL)"
resultado = ramos.invoke({"termo": termo})
for k, v in resultado.items():
    print(f"--- {k} ---\n{v}\n")


--- definicao ---
LCEL (LangChain Expression Language) é uma forma declarativa de combinar componentes de LLM (modelos de linguagem grandes) para criar pipelines complexos, oferecendo flexibilidade e otimização.

--- explicacao_simples ---
Cadeia de prompts (LCEL) é como montar uma linha de montagem para a inteligência artificial, onde cada "estação" é uma instrução específica (prompt) que ela segue para processar informações e construir uma resposta completa.

